# MindStream — CV Emotion Model (Model 1, revised — simple version)
TensorFlow/Keras, MobileNetV2 backbone, FER2013 -> 7-class emotion classifier.

This version deliberately sticks to the same tools used across your other notebooks:
- `ImageDataGenerator` + `flow_from_directory` for loading and augmenting images (like `age_gender_revised.ipynb`
  and `transfer_learning_finetuning.ipynb`), instead of a `tf.data` pipeline with custom `.map()` augmentation layers.
- One backbone (MobileNetV2), frozen first, then unfrozen from a fixed layer index — the same freeze/unfreeze
  pattern as `transfer_learning_finetuning.ipynb` (which unfreezes from `block5_conv1` onward on VGG16).
- Class imbalance handled with plain `class_weight` (a dict passed to `model.fit`) — no focal loss, no custom
  loss classes.
- Label smoothing via Keras's *built-in* `label_smoothing` argument on `CategoricalCrossentropy` — a one-line
  setting, not custom code.
- The same three callbacks as before: `ModelCheckpoint`, `ReduceLROnPlateau`, `EarlyStopping`.

Two-phase training: head warmup (backbone frozen) -> fine-tune (top layers unfrozen).

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from keras.layers import Dense, Dropout, BatchNormalization, GlobalAveragePooling2D, Input
from keras.models import Model
from keras import optimizers, callbacks, losses
from keras.applications import MobileNetV2
from keras.applications.mobilenet_v2 import preprocess_input
from keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))

## Config
Same folder layout as before: `DATA_DIR/train/<class_name>/...` and `DATA_DIR/val/<class_name>/...`.
Edit the paths here if yours differ.

In [ ]:
DATA_DIR = r"E:\Core-AI\DATASETS\FER2013"
CHECKPOINT_DIR = r"E:\Core-AI\MODELS\CV\checkpoints"

# Order matters for FER_CLASSES: this fixes which label index means which emotion, and we pass it
# to flow_from_directory below so the folders get mapped to these exact indices (alphabetical folder
# order would put "neutral" in the wrong slot otherwise).
FER_CLASSES = ["angry", "disgust", "fear", "happy", "sad", "surprise", "neutral"]
NUM_CLASSES = len(FER_CLASSES)

IMG_SIZE = (96, 96)     # up from the raw 48x48 grayscale FER2013 images; MobileNetV2 supports this size natively
BATCH_SIZE = 32

EPOCHS_WARMUP = 15
EPOCHS_FINETUNE = 25
LABEL_SMOOTHING = 0.1

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

train_dir = os.path.join(DATA_DIR, "train")
val_dir = os.path.join(DATA_DIR, "val")

## 1. Data augmentation + generators
Same `ImageDataGenerator` idiom as `age_gender_revised.ipynb` / `transfer_learning_feature_extraction.ipynb`,
just with a fuller set of augmentation knobs (rotation, shifts, shear, zoom, flip, brightness).

Two things worth noting:
- `zoom_range` is doing the job of "random crop" here — zooming in randomly on each image has basically the
  same effect as a random crop, without needing extra crop code.
- Instead of `rescale=1./255`, we pass MobileNetV2's own `preprocess_input` as `preprocessing_function`. That's
  the correct preprocessing for a pretrained MobileNetV2 (it scales pixels to `[-1, 1]`, not `[0, 1]`) and it's
  a single built-in argument swap, not extra code.
- `color_mode='rgb'` on grayscale FER2013 images makes Keras replicate the single channel into 3 automatically,
  so MobileNetV2's ImageNet weights still apply.

In [ ]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.15,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True,
    fill_mode="nearest",
)

val_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    color_mode="rgb",
    classes=FER_CLASSES,
    class_mode="categorical",
    batch_size=BATCH_SIZE,
    shuffle=True,
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    color_mode="rgb",
    classes=FER_CLASSES,
    class_mode="categorical",
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print("Class indices:", train_generator.class_indices)
print("Train samples:", train_generator.samples, "| Val samples:", val_generator.samples)

## 2. Class weights
FER2013 is heavily imbalanced (`disgust` especially). `compute_class_weight` gives each class a weight
inversely proportional to how many samples it has, and we pass that straight into `model.fit(class_weight=...)`
— no custom loss needed for this part.

In [ ]:
class_weights_arr = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(train_generator.classes),
    y=train_generator.classes,
)
class_weight_dict = dict(enumerate(class_weights_arr))
print("Class weights:", {FER_CLASSES[i]: round(w, 3) for i, w in class_weight_dict.items()})

## 3. Model architecture
MobileNetV2 backbone (frozen initially) + a small classifier head — same shape as the original notebook
(GlobalAveragePooling -> BatchNorm -> Dropout -> Dense -> Dropout -> Dense).

In [ ]:
def build_model(input_shape, num_classes):
    base_model = MobileNetV2(input_shape=input_shape, include_top=False, weights="imagenet")
    base_model.trainable = False

    inputs = Input(shape=input_shape)
    x = base_model(inputs, training=False)
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dropout(0.4)(x)
    x = Dense(128, activation="relu")(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation="softmax")(x)

    model = Model(inputs, outputs, name="MindStream_MobileNetV2")
    return model, base_model

model, base_model = build_model(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3), num_classes=NUM_CLASSES)
model.summary()

## 4. Callbacks
Same three as before: best-checkpoint saving, LR reduction on plateau, and early stopping so we don't have
to hand-tune epoch counts.

In [ ]:
checkpoint_path = os.path.join(CHECKPOINT_DIR, "best_emotion_model.keras")

cb_list = [
    callbacks.ModelCheckpoint(checkpoint_path, monitor="val_accuracy", save_best_only=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, verbose=1),
]

## Phase 1 — Train the classification head (backbone frozen)
`label_smoothing=0.1` on `CategoricalCrossentropy` is the built-in Keras way to do label smoothing — it softens
the one-hot targets slightly (e.g. `[0,1,0,...]` becomes `[0.014, 0.9, 0.014, ...]`) so the model doesn't get
overconfident, which tends to generalize better.

In [ ]:
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss=losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=["accuracy"],
)

history_phase1 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_WARMUP,
    class_weight=class_weight_dict,
    callbacks=cb_list,
)

## Phase 2 — Fine-tune MobileNetV2
Same pattern as `transfer_learning_finetuning.ipynb`: unfreeze the backbone, then re-freeze everything up to
a chosen layer so only the top portion trains, at a much lower learning rate. MobileNetV2 has ~155 layers;
unfreezing from layer 100 leaves roughly the top third trainable.

In [ ]:
base_model.trainable = True

fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5),
    loss=losses.CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=["accuracy"],
)

history_phase2 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS_WARMUP + EPOCHS_FINETUNE,
    initial_epoch=history_phase1.epoch[-1] + 1,
    class_weight=class_weight_dict,
    callbacks=cb_list,
)

print(f"\nTraining complete. Best checkpoint saved to:\n  {checkpoint_path}")

## 5. Accuracy / loss curves
Same style of plot as `transfer_learning_finetuning.ipynb` and `Dogs_vs_Cats___CampusX.ipynb`, just stitching
phase 1 and phase 2 together so you can see the jump (or lack of one) when fine-tuning kicks in.

In [ ]:
acc = history_phase1.history["accuracy"] + history_phase2.history["accuracy"]
val_acc = history_phase1.history["val_accuracy"] + history_phase2.history["val_accuracy"]
loss = history_phase1.history["loss"] + history_phase2.history["loss"]
val_loss = history_phase1.history["val_loss"] + history_phase2.history["val_loss"]
switch_epoch = len(history_phase1.history["accuracy"])

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(acc, color="red", label="train")
plt.plot(val_acc, color="blue", label="validation")
plt.axvline(switch_epoch, color="gray", linestyle="--", label="fine-tune starts")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(loss, color="red", label="train")
plt.plot(val_loss, color="blue", label="validation")
plt.axvline(switch_epoch, color="gray", linestyle="--", label="fine-tune starts")
plt.title("Loss")
plt.xlabel("Epoch")
plt.legend()

plt.show()

## 6. Evaluation
Classification report + confusion matrix on the held-out val set, using the best saved checkpoint. Look at
the per-class rows for `disgust` in particular — that's the class the class weights are working hardest for.

In [ ]:
best_model = tf.keras.models.load_model(checkpoint_path)

val_generator.reset()
steps = int(np.ceil(val_generator.samples / BATCH_SIZE))
preds = best_model.predict(val_generator, steps=steps, verbose=0)

y_pred = np.argmax(preds, axis=1)[: val_generator.samples]
y_true = val_generator.classes

print(classification_report(y_true, y_pred, target_names=FER_CLASSES))
print(confusion_matrix(y_true, y_pred))

## Optional: focal loss instead of class_weight
Skip this unless `disgust` is still noticeably worse than the other classes after a full run with the setup
above. It's the same idea as `class_weight` — pay more attention to hard/rare examples — but weights each
prediction by how wrong it currently is, not just by class frequency. It's a plain function (no custom class),
so you can compare it directly against `CategoricalCrossentropy`.

To use it: replace `loss=losses.CategoricalCrossentropy(...)` with `loss=focal_loss(gamma=2.0)` in the
`model.compile(...)` calls above, and drop `class_weight=class_weight_dict` from `model.fit(...)` (using both
at once double-corrects for imbalance).

In [ ]:
def focal_loss(gamma=2.0):
    def loss_fn(y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        cross_entropy = -y_true * tf.math.log(y_pred)
        weight = y_true * tf.pow(1 - y_pred, gamma)
        return tf.reduce_sum(weight * cross_entropy, axis=-1)
    return loss_fn